# ⏱️ Survival Analysis
**ISLP Chapter 11 · Pattern Recognition for the Rest of Us**

> Survival analysis answers questions like: how long until a patient relapses? When will a customer churn? It handles a unique challenge — censored observations: people who haven't experienced the event yet when the study ends.

### What you'll learn
- Censoring: why standard regression fails for time-to-event data
- Kaplan-Meier estimator: non-parametric survival curve
- Log-rank test: comparing survival between groups
- Cox proportional hazards model: regression for survival data
- Hazard ratio interpretation

### Dataset: ROSSI recidivism + simulated patient data
### Time: ~60 minutes

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
plt.rcParams.update({'figure.facecolor':'white','axes.facecolor':'#f8f9fa','axes.grid':True,'grid.alpha':0.4,'axes.spines.top':False,'axes.spines.right':False,'font.size':11})
print('✓ Setup complete')
!pip install lifelines -q
from lifelines import KaplanMeierFitter, CoxPHFitter
from lifelines.statistics import logrank_test
from lifelines.datasets import load_rossi
import warnings; warnings.filterwarnings('ignore')

# Rossi recidivism dataset: time until re-arrest after release from prison
rossi = load_rossi()
print(f'Rossi dataset: {rossi.shape}')
print(rossi.head())
print(f"\nEvent rate: {rossi['arrest'].mean():.1%} arrested")
print(f"Duration range: {rossi['week'].min()}–{rossi['week'].max()} weeks")

## 🎯 Part 1 — What is Censoring?

In time-to-event data:
- Some subjects experience the event (arrested, died, churned)
- Others are **censored** — we know they survived UNTIL a certain time, but not what happened after
  - Study ended before they experienced the event
  - Lost to follow-up
  - Withdrew from study

**Why standard regression fails:** A censored observation isn't a missing value — it contains real information (the person survived for at least X weeks). Ignoring it or treating it as missing biases estimates.

**Survival analysis** handles censoring properly.

In [ ]:
# Visualize censoring
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Show 20 individual timelines
np.random.seed(42)
sample_idx = np.random.choice(len(rossi), 25, replace=False)
for i, idx in enumerate(sample_idx):
    row = rossi.iloc[idx]
    color = '#e85d20' if row['arrest']==1 else '#1e5fa8'
    marker = 'x' if row['arrest']==1 else 'o'
    axes[0].plot([0, row['week']], [i, i], color=color, lw=1.5, alpha=0.8)
    axes[0].plot(row['week'], i, marker=marker, color=color, markersize=7)
axes[0].set_xlabel('Weeks'); axes[0].set_ylabel('Subject')
axes[0].set_title('Event timelines\n× = arrested (event), o = censored')
from matplotlib.lines import Line2D
legend = [Line2D([0],[0],color='#e85d20',marker='x',label='Arrested'),
          Line2D([0],[0],color='#1e5fa8',marker='o',label='Censored')]
axes[0].legend(handles=legend)

# Histogram of observation times by event status
for arrested, color, label in [(1,'#e85d20','Arrested'),(0,'#1e5fa8','Censored')]:
    subset = rossi[rossi['arrest']==arrested]['week']
    axes[1].hist(subset, bins=20, alpha=0.6, color=color, label=f'{label} (n={len(subset)})', edgecolor='white')
axes[1].set_xlabel('Weeks'); axes[1].set_ylabel('Count')
axes[1].set_title('Distribution of Follow-up Times')
axes[1].legend()
plt.tight_layout(); plt.show()

In [ ]:
# Kaplan-Meier survival curve — overall
kmf = KaplanMeierFitter()
kmf.fit(rossi['week'], event_observed=rossi['arrest'], label='All subjects')

fig, ax = plt.subplots(figsize=(9, 5))
kmf.plot_survival_function(ax=ax, color='#1e5fa8', linewidth=2.5, ci_show=True, ci_alpha=0.15)
ax.set_xlabel('Weeks after release')
ax.set_ylabel('Probability of not being re-arrested')
ax.set_title('Kaplan-Meier Survival Curve — Recidivism Study')
ax.axhline(0.5, color='#888', lw=1, ls='--', label='Median survival')
median = kmf.median_survival_time_
if not np.isinf(median):
    ax.axvline(median, color='#888', lw=1, ls='--')
    ax.text(median+1, 0.52, f'Median = {median:.0f} weeks', fontsize=9)
ax.legend()
plt.tight_layout(); plt.show()
print(f'\nMedian survival time: {kmf.median_survival_time_:.1f} weeks')
print(f'52-week survival estimate: {kmf.survival_function_at_times([52]).values[0]:.3f}')

In [ ]:
# Compare groups: financial aid vs no financial aid
fig, ax = plt.subplots(figsize=(9, 5))
for group, color, label in [(1,'#1a7a45','Received financial aid'),(0,'#e85d20','No financial aid')]:
    kmf_g = KaplanMeierFitter()
    mask = rossi['fin']==group
    kmf_g.fit(rossi.loc[mask,'week'], event_observed=rossi.loc[mask,'arrest'], label=label)
    kmf_g.plot_survival_function(ax=ax, color=color, linewidth=2.5, ci_show=True, ci_alpha=0.15)
ax.set_xlabel('Weeks after release'); ax.set_ylabel('Probability of not being re-arrested')
ax.set_title('Kaplan-Meier by Financial Aid Status')
ax.legend()
plt.tight_layout(); plt.show()

# Log-rank test
results = logrank_test(
    rossi.loc[rossi['fin']==1,'week'], rossi.loc[rossi['fin']==0,'week'],
    event_observed_A=rossi.loc[rossi['fin']==1,'arrest'],
    event_observed_B=rossi.loc[rossi['fin']==0,'arrest']
)
print(f'Log-rank test: p-value = {results.p_value:.4f}')
print('Financial aid significantly reduces recidivism' if results.p_value < 0.05 else 'No significant difference')

In [ ]:
# Cox Proportional Hazards Model
cox_features = ['fin','age','race','wexp','mar','paro','prio']
cph = CoxPHFitter()
cph.fit(rossi[cox_features + ['week','arrest']], duration_col='week', event_col='arrest')
cph.print_summary()

# Plot hazard ratios
fig, ax = plt.subplots(figsize=(8, 5))
cph.plot(ax=ax)
ax.set_title('Cox Model: Hazard Ratios (HR > 1 = higher risk of arrest)')
ax.axvline(0, color='black', lw=0.8, ls='--')
plt.tight_layout(); plt.show()
print('\n📌 HR < 0 (log scale) = protective factor, HR > 0 = risk factor')
print(f"   'prio' (prior arrests): HR = {np.exp(cph.params_['prio']):.2f} — each prior arrest increases risk by {(np.exp(cph.params_['prio'])-1)*100:.0f}%")

In [ ]:
# Exercise workspace
# Task 1: Compare survival curves by age group (< 30 vs >= 30)
# YOUR CODE HERE

# Task 2: Build a Cox model using only the 3 most significant predictors
# Compare its concordance index to the full model
# YOUR CODE HERE

# Task 3: Predict individual survival curves for two specific profiles:
# Profile A: young (25), prior arrests=0, financial aid
# Profile B: older (40), prior arrests=3, no financial aid
profile_A = pd.DataFrame({'fin':1,'age':25,'race':1,'wexp':1,'mar':0,'paro':1,'prio':0}, index=[0])
profile_B = pd.DataFrame({'fin':0,'age':40,'race':1,'wexp':0,'mar':0,'paro':0,'prio':3}, index=[0])
# YOUR CODE HERE

In [ ]:
# @title 📝 Quiz — Survival Analysis
# @markdown Answer each question in the box below, then run the AI grading cell.

# @markdown **Q1:** What is censoring in survival analysis?
# @markdown **Q2:** What does the Kaplan-Meier curve show?
# @markdown **Q3:** What does the log-rank test compare?
# @markdown **Q4:** What does a hazard ratio of 2.0 mean in a Cox model?
# @markdown **Q5:** Why can't you use standard linear regression for time-to-event outcomes?

q1 = "" # @param {type:"string", placeholder:"your answer"}
q2 = "" # @param {type:"string", placeholder:"your answer"}
q3 = "" # @param {type:"string", placeholder:"your answer"}
q4 = "" # @param {type:"string", placeholder:"your answer"}
q5 = "" # @param {type:"string", placeholder:"your answer"}

answers = {"q1": q1, "q2": q2, "q3": q3, "q4": q4, "q5": q5}
answered = sum(1 for v in answers.values() if str(v).strip())
print(f"  {answered}/5 answered — run the AI grading cell below")

### 📤 Submit your results

In [ ]:
_NB_NAME_ = "Survival Analysis"
# @title 🤖 AI Feedback — enter your username and click ▶ Run
# @markdown **No API key needed.** AI grading runs free inside your Colab session.
# @markdown
# @markdown Enter your GitHub username, then click ▶ Run for question-by-question feedback.

GITHUB_USERNAME = "" # @param {type:"string", placeholder:"e.g. jsmith42"}

# ── runs automatically — do not edit below ───────────────────
import json, textwrap, re as _re, time
_NB_TITLE = globals().get("_NB_NAME_", "this notebook")

def _get_quiz_questions():
    """Pull question text from the quiz cell @markdown lines."""
    try:
        import ipynbname
        nb_path = ipynbname.path()
        with open(nb_path) as f:
            nb = json.load(f)
        for cell in nb["cells"]:
            src = "".join(cell.get("source", []))
            if "@title" in src and "Quiz" in src:
                qs = re.findall(r"@markdown \*\*Q\d+:\*\* (.+)", src)
                if qs: return qs
    except Exception:
        pass
    return []

def _call_gemini(prompt, max_retries=3):
    """Call Gemini with retry on 429 rate limit."""
    last_err = None
    for attempt in range(max_retries):
        try:
            import google.generativeai as genai
            import google.auth, google.auth.transport.requests
            creds, _ = google.auth.default()
            creds.refresh(google.auth.transport.requests.Request())
            genai.configure(credentials=creds)
            model  = genai.GenerativeModel("gemini-2.0-flash")
            result = model.generate_content(
                prompt,
                generation_config={"max_output_tokens": 1500, "temperature": 0.3}
            )
            return result.text, "Gemini via Colab"
        except Exception as e:
            last_err = str(e)
            if "429" in str(e) or "quota" in str(e).lower():
                wait = 2 ** attempt
                print(f"  Rate limit hit — waiting {wait}s before retry {attempt+1}/{max_retries}...")
                time.sleep(wait)
                continue
            break
    # Try stored key as fallback
    try:
        from google.colab import userdata
        key = userdata.get("GEMINI_API_KEY")
        if key and len(key) > 10:
            import google.generativeai as genai
            genai.configure(api_key=key)
            model  = genai.GenerativeModel("gemini-2.0-flash")
            result = model.generate_content(prompt)
            return result.text, "Gemini via key"
    except Exception:
        pass
    return None, last_err

def _build_prompt(answers_dict, nb_name, questions):
    answered  = {k: v for k, v in answers_dict.items() if str(v).strip()}
    n_total   = len(answers_dict)
    n_done    = len(answered)

    # Pair each question with the student answer
    qa_pairs = ""
    for i, (k, v) in enumerate(answers_dict.items(), 1):
        q_text = questions[i-1] if i-1 < len(questions) else f"Question {i}"
        a_text = str(v).strip() if str(v).strip() else "(no answer)"
        qa_pairs += f"Q{i}: {q_text}\nA{i}: {a_text}\n\n"

    return f"""You are a TA grading a student quiz for the "{nb_name}" notebook in a data science course called "Data Pattern Recognition for the Rest of Us" (based on ISLP and business analytics).

{qa_pairs.strip()}

For EACH question:
- Decide if the answer is CORRECT, PARTIALLY CORRECT, or INCORRECT
- A short paraphrase or reasonable approximation counts as CORRECT — do not require exact wording
- For INCORRECT or PARTIAL: name the specific concept they need to review (1 sentence)
- For CORRECT: brief confirmation of what they got right (1 sentence)

Then give an overall summary.

Respond ONLY in this exact JSON format (no markdown fences, no extra text):
{{
  "questions": [
    {{
      "q": 1,
      "status": "<CORRECT|PARTIAL|INCORRECT>",
      "comment": "<one specific sentence>"
    }}
  ],
  "quiz_score": <integer 0-{n_total}>,
  "grade": "<Excellent|Good|Needs Review|Incomplete>",
  "summary": "<2 sentences overall: strengths and what to revisit>",
  "review_tip": "<one specific concept, chapter, or notebook to review if they struggled — or 'Great work!' if excellent>"
}}

Scoring guide: CORRECT=1pt, PARTIAL=0.5pt (round to nearest int), INCORRECT=0pt."""

def _show(result, username, nb_name, source, questions):
    STATUS_ICON  = {"CORRECT": "\u2705", "PARTIAL": "\u26a0\ufe0f", "INCORRECT": "\u274c"}
    STATUS_COLOR = {"CORRECT": "\033[92m", "PARTIAL": "\033[93m", "INCORRECT": "\033[91m"}
    R = "\033[0m"
    grade = result.get("grade", "?")
    GRADE_COLOR = {"Excellent":"\033[92m","Good":"\033[94m",
                   "Needs Review":"\033[93m","Incomplete":"\033[91m"}
    GC = GRADE_COLOR.get(grade, "")
    n  = len(answers)
    qs = result.get("quiz_score", 0)
    bar = "\u2588"*qs + "\u2591"*(n-qs)

    print("\n" + "\u2550"*60)
    print(f"  \U0001f916 AI Feedback \u2014 {nb_name}")
    if source: print(f"  Powered by   {source}")
    print("\u2550"*60)
    print(f"  Student  : {'@'+username if username else '\u26a0 set GITHUB_USERNAME above'}")
    print(f"  Grade    : {GC}{grade}{R}")
    print(f"  Score    : {qs}/{n}  [{bar}]")
    print()

    # Question-by-question breakdown
    q_results = result.get("questions", [])
    if q_results:
        print("  \u2500"*29)
        for qr in q_results:
            idx    = qr.get("q", 0) - 1
            status = qr.get("status", "?")
            icon   = STATUS_ICON.get(status, "\u2753")
            color  = STATUS_COLOR.get(status, "")
            comment= qr.get("comment", "")
            q_text = questions[idx] if idx < len(questions) else f"Question {qr.get('q','?')}"
            # Truncate long question text for display
            q_short = q_text[:55] + "..." if len(q_text) > 55 else q_text
            print(f"  {icon} {color}Q{qr.get('q','?')} {status}{R}")
            print(f"     {q_short}")
            if comment:
                for line in textwrap.wrap(comment, 54):
                    print(f"     \u2192 {line}")
            print()

    # Summary
    summary = result.get("summary", "")
    if summary:
        print("  \u2500"*29)
        print("  Overall:")
        for line in textwrap.wrap(summary, 56):
            print(f"  {line}")

    # Review tip
    tip = result.get("review_tip", "")
    if tip and tip != "Great work!":
        print()
        for line in textwrap.wrap(f"\U0001f4d6 Review: {tip}", 56):
            print(f"  {line}")
    elif tip == "Great work!":
        print()
        print("  \U0001f4d6 Great work! Keep going.")

    print("\u2550"*60 + "\n")

def _fallback_grade(answers_dict):
    """Smart fallback — grade on completion only, no length penalty."""
    n  = len(answers_dict)
    nd = sum(1 for v in answers_dict.values() if str(v).strip())
    if nd == 0:
        return {"quiz_score":0,"grade":"Incomplete",
                "questions":[],
                "summary":"No answers provided — fill in the quiz above.",
                "review_tip":"Complete the quiz and re-run for AI feedback."}
    elif nd == n:
        return {"quiz_score":n,"grade":"Good",
                "questions":[],
                "summary":f"All {n} questions answered. AI grading was unavailable — re-run to get question-by-question feedback.",
                "review_tip":"Re-run this cell in a few minutes for detailed AI feedback."}
    else:
        return {"quiz_score":nd,"grade":"Needs Review",
                "questions":[],
                "summary":f"{nd}/{n} questions answered. Complete the remaining {n-nd} and re-run.",
                "review_tip":"Answer all questions for full feedback."}

# ── Main ──────────────────────────────────────────────────────
if "answers" not in globals():
    print("\u26a0  Run the quiz cell above first, then re-run this cell.")
else:
    nd       = sum(1 for v in answers.values() if str(v).strip())
    username = GITHUB_USERNAME.strip()
    questions = _get_quiz_questions()

    print(f"\n  Notebook : {_NB_TITLE}  \u2022  {nd}/{len(answers)} answered")
    if username: print(f"  Student  : @{username}")
    print(f"  Grading  : please wait...\n")

    prompt     = _build_prompt(answers, _NB_TITLE, questions)
    raw, src   = _call_gemini(prompt)

    if raw:
        try:
            clean  = _re.sub(r"```(?:json)?\s*|```","",raw).strip()
            result = json.loads(clean)
        except Exception:
            nd2    = sum(1 for v in answers.values() if str(v).strip())
            result = {"quiz_score":nd2,"grade":"Good" if nd2==len(answers) else "Needs Review",
                      "questions":[],"summary":raw[:400],"review_tip":""}
    else:
        if src: print(f"  \u26a0 Gemini unavailable ({src[:60]}) \u2014 showing completion feedback\n")
        result = _fallback_grade(answers)

    _show(result, username, _NB_TITLE, src if raw else None, questions)
    print("  \U0001f4be  File \u2192 Save a copy in GitHub \u2192 your fork\n")


## 📚 Further Reading
- [ISLP Ch 11](https://intro-stat-learning.github.io/ISLP/) — Survival analysis
- [lifelines docs](https://lifelines.readthedocs.io/)
- [Cox model interpretation guide](https://lifelines.readthedocs.io/en/latest/Survival%20Regression.html)

---
*Pattern Recognition for the Rest of Us · [ladataanalytics.github.io/pattern-recognition-notebooks](https://ladataanalytics.github.io/pattern-recognition-notebooks)*